# LeetCode 3921 — Score Validator bằng Python

Link đề: https://leetcode.com/problems/score-validator/ · Độ khó: **Easy** · Chủ đề: Array, String, Simulation

Skeleton LeetCode:

```python
class Solution:
    def scoreValidator(self, events: List[str]) -> List[int]:
        ...
```

> Ghi chú: tên bài, ví dụ, constraints và chữ ký hàm đã được đối chiếu qua leetcode.doocs.org (Biweekly Contest 182, câu 1, Easy, không Premium); phần phát biểu đề được viết lại bằng lời của mình nên cần đối chiếu từng chữ với đề gốc nếu muốn chắc chắn.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mô phỏng việc ghi điểm của một trận đấu (kiểu cricket) từ danh sách sự kiện dạng chuỗi: chữ số cộng điểm, `"W"` tăng bộ đếm, `"WD"`/`"NB"` cộng 1 điểm; **dừng** khi bộ đếm chạm `10`.
2. Biết viết bài **mô phỏng (simulation)** với `for`, `if / elif / else` và `break` để dừng sớm.
3. Biết dùng `dict` làm bảng điểm, `int(token)` để đổi chuỗi chữ số thành số, `functools.reduce`, `Counter`.
4. Viết 5 cách khác nhau và đối chiếu bằng test ngẫu nhiên có `random.seed` (kể cả các dãy có hơn 10 lần `"W"`).
5. Tránh lỗi: không dừng ở 10, dừng sai thời điểm (`> 10`), nhầm `"WD"` với `"W"`, gọi `int("WD")`, quên `"NB"`.

## 2. Đề bài & ví dụ

Cho mảng chuỗi `events`. Ban đầu `score = 0` và `counter = 0`. Xử lý các sự kiện **từ trái sang phải**:

- Chuỗi là một chữ số (`"0"`, `"1"`, `"2"`, `"3"`, `"4"`, `"6"`): cộng giá trị đó vào `score`.
- `"W"`: tăng `counter` thêm 1, **không** đổi `score`.
- `"WD"` hoặc `"NB"`: cộng thêm `1` vào `score`.

Dừng xử lý khi đã xử lý hết mảng **hoặc** khi `counter` chạm `10` (các sự kiện còn lại bị bỏ qua). Trả về `[score, counter]`.

| Ví dụ | events | Output | Giải thích |
|---|---|---|---|
| 1 | `["1","4","W","6","WD"]` | `[12,1]` | 1 + 4 = 5; `W` làm counter = 1; + 6 = 11; `WD` + 1 = 12 |
| 2 | `["WD","NB","0","4","4"]` | `[10,0]` | `WD`, `NB` mỗi cái 1 điểm -> 2; `0` không đổi; `4 + 4` -> 10; counter vẫn 0 |
| 3 | 11 lần `"W"` | `[0,10]` | Counter đạt 10 sau lần thứ 10, dừng; lần thứ 11 bị bỏ qua |

Constraints: `1 <= events.length <= 1000`; mỗi `events[i]` thuộc `{"0","1","2","3","4","6","W","WD","NB"}`.

In [1]:
examples = [["1", "4", "W", "6", "WD"], ["WD", "NB", "0", "4", "4"], ["W"] * 11]
for ev in examples:
    print(ev[:6], "..." if len(ev) > 6 else "", "độ dài", len(ev))

['1', '4', 'W', '6', 'WD']  độ dài 5
['WD', 'NB', '0', '4', '4']  độ dài 5
['W', 'W', 'W', 'W', 'W', 'W'] ... độ dài 11


## 3. Kiến thức nền

### 3.1. Chuỗi chữ số -> số nguyên với `int`

`int("4")` là `4`. Nhưng `int("W")` hoặc `int("WD")` gây `ValueError`. Với các token có thể là chữ cái, cần kiểm tra trước (`token.isdigit()`) hoặc dùng bảng tra cứu.

In [2]:
print(int("4"), int("0"), "6".isdigit(), "WD".isdigit(), "W".isdigit())
try:
    int("WD")
except ValueError as e:
    print("ValueError:", e)

4 0 True False False
ValueError: invalid literal for int() with base 10: 'WD'


### 3.2. `if / elif / else` và so sánh chuỗi

So sánh chuỗi bằng `==`: `token == "W"`; `token in ("WD", "NB")` kiểm tra thuộc tập. Thứ tự các nhánh: `"W"` và `"WD"` là hai chuỗi **khác nhau** (không dùng `startswith("W")` vì sẽ nhầm).

In [3]:
for token in ["W", "WD", "NB", "6"]:
    print(token, token == "W", token in ("WD", "NB"), token.startswith("W"))

W True False True
WD False True True
NB False True False
6 False False False


### 3.3. `break` — dừng vòng lặp sớm

`break` thoát khỏi vòng `for` ngay lập tức. Bài này cần dừng **ngay sau khi** `counter` chạm `10`, không xử lý sự kiện nào nữa.

In [4]:
for i in range(10):
    if i == 3:
        break
    print("i =", i)
print("đã thoát vòng lặp")

i = 0
i = 1
i = 2
đã thoát vòng lặp


### 3.4. `dict` làm bảng điểm

`SCORE = {"0": 0, "1": 1, ..., "WD": 1, "NB": 1, "W": 0}` ánh xạ mỗi token tới số điểm cộng thêm, tránh chuỗi `if` dài. Còn `"W"` đặc biệt vì còn tăng `counter`.

In [5]:
SCORE = {"0": 0, "1": 1, "2": 2, "3": 3, "4": 4, "6": 6, "W": 0, "WD": 1, "NB": 1}
print(SCORE["4"], SCORE["WD"], SCORE["W"], SCORE.get("5", "không có"))

4 1 0 không có


### 3.5. `functools.reduce` và `Counter`

`reduce(f, iterable, init)` gấp dần: `f(f(init, a), b)`. Dùng để cập nhật một "trạng thái" `(score, counter)`. `Counter(list)` đếm số lần của mỗi token.

In [6]:
from functools import reduce
from collections import Counter
print(reduce(lambda acc, x: acc + x, [1, 2, 3], 0))
print(Counter(["1", "W", "W", "WD"]))

6
Counter({'W': 2, '1': 1, 'WD': 1})


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Đây là bài **mô phỏng**: giữ hai biến `score`, `counter`, đọc từng sự kiện và áp dụng đúng luật. Điểm khó duy nhất là **điều kiện dừng**: ngay khi `counter == 10` phải thoát (kể cả khi còn sự kiện chứa điểm).

Tính tay ví dụ 1 `["1","4","W","6","WD"]`:

| bước | token | score | counter | ghi chú |
|---|---|---|---|---|
| 0 | | 0 | 0 | khởi tạo |
| 1 | `"1"` | 1 | 0 | +1 |
| 2 | `"4"` | 5 | 0 | +4 |
| 3 | `"W"` | 5 | 1 | counter +1 |
| 4 | `"6"` | 11 | 1 | +6 |
| 5 | `"WD"` | 12 | 1 | +1 (WD là điểm thưởng, không phải W) |

Kết quả `[12, 1]`. Ví dụ 3: `W` lặp 11 lần: counter đạt `10` ở lần thứ 10 và ta dừng, nên lần thứ 11 không được tính.

In [7]:
def by_hand(events):
    score = counter = 0
    for step, t in enumerate(events, 1):
        if t == "W":
            counter += 1
        elif t in ("WD", "NB"):
            score += 1
        else:
            score += int(t)
        print(f"bước {step}: {t!r:5} score={score:2d} counter={counter}")
        if counter == 10:
            print("counter chạm 10 -> dừng")
            break
    return [score, counter]

print(by_hand(["1", "4", "W", "6", "WD"]))
print(by_hand(["W"] * 11)[-2:])

bước 1: '1'   score= 1 counter=0
bước 2: '4'   score= 5 counter=0
bước 3: 'W'   score= 5 counter=1
bước 4: '6'   score=11 counter=1
bước 5: 'WD'  score=12 counter=1
[12, 1]
bước 1: 'W'   score= 0 counter=1
bước 2: 'W'   score= 0 counter=2
bước 3: 'W'   score= 0 counter=3
bước 4: 'W'   score= 0 counter=4
bước 5: 'W'   score= 0 counter=5
bước 6: 'W'   score= 0 counter=6
bước 7: 'W'   score= 0 counter=7
bước 8: 'W'   score= 0 counter=8
bước 9: 'W'   score= 0 counter=9
bước 10: 'W'   score= 0 counter=10
counter chạm 10 -> dừng
[0, 10]


## 5. Cách 1 — Brute Force / baseline

Mô phỏng trực tiếp bằng `for` và `if / elif / else`, có `break` khi `counter == 10`. Đây là cách trực tiếp nhất và cũng đã tối ưu (`O(n)`, mỗi sự kiện xử lý một lần).

"Nút thắt": chuỗi `if` dễ sai khi có nhiều loại token; các cách sau đưa luật vào bảng hoặc tách việc "tìm điểm dừng" khỏi việc "tính điểm".

In [8]:
class SolutionBrute:
    def scoreValidator(self, events):
        score = counter = 0
        for t in events:
            if t == "W":
                counter += 1
            elif t == "WD" or t == "NB":
                score += 1
            else:
                score += int(t)
            if counter == 10:
                break
        return [score, counter]

print(SolutionBrute().scoreValidator(["1", "4", "W", "6", "WD"]), SolutionBrute().scoreValidator(["WD", "NB", "0", "4", "4"]), SolutionBrute().scoreValidator(["W"] * 11))

[12, 1] [10, 0] [0, 10]


**Độ phức tạp:** Time `O(n)` (có thể dừng sớm), Space `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

Tách bài toán thành hai việc độc lập:

1. **Điểm dừng:** tìm `cut` = vị trí (đếm từ 0) ngay **sau** sự kiện `"W"` thứ 10; nếu không có đủ 10 sự kiện `"W"` thì `cut = len(events)`.
2. **Tính toán:** trong `events[:cut]`: `counter` là số `"W"`, `score` là tổng điểm của các token còn lại.

**Invariant** của mô phỏng: sau khi xử lý `k` sự kiện đầu (mà chưa dừng), `score` bằng tổng điểm của `events[:k]` theo luật và `counter` bằng số `"W"` trong `events[:k]`. Ta dừng ở `k` nhỏ nhất thỏa `counter == 10`.

Hai quy tắc dễ nhầm: (1) `"WD"` là **điểm thưởng** (`score + 1`) chứ **không** làm tăng `counter`; (2) khi `counter` chạm `10` ta dừng **ngay**, không xử lý sự kiện kế (kể cả `"WD"` hay chữ số).

## 7. Các cách tiếp theo

### Cách 2 — Bảng điểm `dict`

`SCORE` ánh xạ token -> điểm cộng; riêng `"W"` thì tăng `counter`.

In [9]:
SCORE = {"0": 0, "1": 1, "2": 2, "3": 3, "4": 4, "6": 6, "W": 0, "WD": 1, "NB": 1}

class SolutionDict:
    def scoreValidator(self, events):
        score = counter = 0
        for t in events:
            score += SCORE[t]
            if t == "W":
                counter += 1
                if counter == 10:
                    break
        return [score, counter]

print(SolutionDict().scoreValidator(["1", "4", "W", "6", "WD"]), SolutionDict().scoreValidator(["WD", "NB", "0", "4", "4"]), SolutionDict().scoreValidator(["W"] * 11))

[12, 1] [10, 0] [0, 10]


**Độ phức tạp:** Time `O(n)`, Space `O(1)`.

### Cách 3 — Tìm điểm dừng, rồi cộng trên đoạn cắt

Trước hết tìm `cut` (sau sự kiện `"W"` thứ 10). Rồi `score = sum(SCORE[t] for t in events[:cut])` và `counter = số "W" trong events[:cut]`.

In [10]:
class SolutionCut:
    def scoreValidator(self, events):
        cut, ws = len(events), 0
        for i, t in enumerate(events):
            if t == "W":
                ws += 1
                if ws == 10:
                    cut = i + 1
                    break
        part = events[:cut]
        return [sum(SCORE[t] for t in part), sum(1 for t in part if t == "W")]

print(SolutionCut().scoreValidator(["1", "4", "W", "6", "WD"]), SolutionCut().scoreValidator(["WD", "NB", "0", "4", "4"]), SolutionCut().scoreValidator(["W"] * 11))

[12, 1] [10, 0] [0, 10]


**Độ phức tạp:** Time `O(n)`, Space `O(n)` cho lát cắt.

### Cách 4 — `reduce` với trạng thái bị "đóng băng" khi dừng

Trạng thái `(score, counter)`. Hàm gộp: nếu `counter == 10` thì trả nguyên trạng thái (bỏ qua mọi sự kiện sau). `reduce` không thoát sớm nhưng kết quả đúng.

In [11]:
from functools import reduce

class SolutionReduce:
    def scoreValidator(self, events):
        def step(state, t):
            score, counter = state
            if counter == 10:               # đã dừng: đóng băng trạng thái
                return state
            if t == "W":
                return (score, counter + 1)
            return (score + SCORE[t], counter)
        score, counter = reduce(step, events, (0, 0))
        return [score, counter]

print(SolutionReduce().scoreValidator(["1", "4", "W", "6", "WD"]), SolutionReduce().scoreValidator(["WD", "NB", "0", "4", "4"]), SolutionReduce().scoreValidator(["W"] * 11))

[12, 1] [10, 0] [0, 10]


**Độ phức tạp:** Time `O(n)` (không thoát sớm), Space `O(1)` phụ.

### Cách 5 — `Counter` trên đoạn đã cắt

Sau khi tìm `cut` như Cách 3, đếm token bằng `Counter` rồi tính: `counter = c["W"]`; `score = Σ int(chữ số) × số lần + c["WD"] + c["NB"]`. Cho thấy bài chỉ cần "đếm" khi đã biết điểm dừng.

In [12]:
from collections import Counter

class SolutionCounter:
    def scoreValidator(self, events):
        seen_w, cut = 0, len(events)
        for i, t in enumerate(events):
            if t == "W":
                seen_w += 1
                if seen_w == 10:
                    cut = i + 1
                    break
        c = Counter(events[:cut])
        score = sum(int(tok) * k for tok, k in c.items() if tok.isdigit()) + c["WD"] + c["NB"]
        return [score, c["W"]]

print(SolutionCounter().scoreValidator(["1", "4", "W", "6", "WD"]), SolutionCounter().scoreValidator(["WD", "NB", "0", "4", "4"]), SolutionCounter().scoreValidator(["W"] * 11))

[12, 1] [10, 0] [0, 10]


**Độ phức tạp:** Time `O(n)`, Space `O(n)`.

## 8. TRACE từng bước

Trace Cách 1 trên một dãy có điều kiện dừng: `events = ["W","W","W","W","W","W","W","W","W","W","6","WD"]` (10 lần `W`, rồi `6`, `WD`):

| bước | token | score | counter | hành động |
|---|---|---|---|---|
| 1..9 | `W` | 0 | 1..9 | tăng counter |
| 10 | `W` | 0 | 10 | counter chạm 10 -> **dừng** |
| 11, 12 | `6`, `WD` | (không xử lý) | | bị bỏ qua |

Kết quả `[0, 10]` (điểm `6` và `WD` không được tính).

In [13]:
def trace(events):
    score = counter = 0
    for step, t in enumerate(events, 1):
        if t == "W":
            counter += 1
        elif t in ("WD", "NB"):
            score += 1
        else:
            score += int(t)
        print(f"bước {step:2d}: {t!r:5} -> score={score:2d} counter={counter:2d}")
        if counter == 10:
            print(f"  counter = 10 -> dừng, bỏ qua {len(events) - step} sự kiện còn lại")
            break
    return [score, counter]

assert trace(["W"] * 10 + ["6", "WD"]) == [0, 10]
assert trace(["1", "4", "W", "6", "WD"]) == [12, 1]

bước  1: 'W'   -> score= 0 counter= 1
bước  2: 'W'   -> score= 0 counter= 2
bước  3: 'W'   -> score= 0 counter= 3
bước  4: 'W'   -> score= 0 counter= 4
bước  5: 'W'   -> score= 0 counter= 5
bước  6: 'W'   -> score= 0 counter= 6
bước  7: 'W'   -> score= 0 counter= 7
bước  8: 'W'   -> score= 0 counter= 8
bước  9: 'W'   -> score= 0 counter= 9
bước 10: 'W'   -> score= 0 counter=10
  counter = 10 -> dừng, bỏ qua 2 sự kiện còn lại
bước  1: '1'   -> score= 1 counter= 0
bước  2: '4'   -> score= 5 counter= 0
bước  3: 'W'   -> score= 5 counter= 1
bước  4: '6'   -> score=11 counter= 1
bước  5: 'WD'  -> score=12 counter= 1


In [14]:
def trace_cut(events):
    ws = 0
    for i, t in enumerate(events):
        if t == "W":
            ws += 1
            print(f"vị trí {i}: W thứ {ws}")
            if ws == 10:
                print(f"điểm dừng: cắt tại {i + 1}")
                return i + 1
    print("không đủ 10 W: không cắt, dùng toàn bộ", len(events))
    return len(events)

assert trace_cut(["W"] * 11) == 10
assert trace_cut(["1", "W"]) == 2

vị trí 0: W thứ 1
vị trí 1: W thứ 2
vị trí 2: W thứ 3
vị trí 3: W thứ 4
vị trí 4: W thứ 5
vị trí 5: W thứ 6
vị trí 6: W thứ 7
vị trí 7: W thứ 8
vị trí 8: W thứ 9
vị trí 9: W thứ 10
điểm dừng: cắt tại 10
vị trí 1: W thứ 1
không đủ 10 W: không cắt, dùng toàn bộ 2


## 9. Code nộp LeetCode

Dùng Cách 1 (mô phỏng trực tiếp): rõ ràng, `O(n)`, `O(1)` bộ nhớ.

In [15]:
from typing import List, Optional


class Solution:
    def scoreValidator(self, events: List[str]) -> List[int]:
        score = counter = 0
        for t in events:
            if t == "W":
                counter += 1                  # chỉ tăng counter, không cộng điểm
            elif t == "WD" or t == "NB":
                score += 1                    # điểm thưởng
            else:
                score += int(t)               # chữ số "0".."6"
            if counter == 10:                 # dừng ngay khi counter chạm 10
                break
        return [score, counter]


print(Solution().scoreValidator(["1", "4", "W", "6", "WD"]), Solution().scoreValidator(["WD", "NB", "0", "4", "4"]), Solution().scoreValidator(["W"] * 11))

[12, 1] [10, 0] [0, 10]


## 10. Vì sao thuật toán đúng

1. **Luật cập nhật.** Mỗi token thuộc đúng một trong ba loại (`"W"`, `"WD"/"NB"`, chữ số), và nhánh tương ứng cộng đúng lượng quy định vào `score` hoặc `counter`. Nhánh `else` chỉ nhận chữ số (theo constraints), nên `int(t)` an toàn.
2. **Điều kiện dừng.** Ta kiểm tra `counter == 10` **sau** mỗi cập nhật; vì `counter` tăng đúng 1 mỗi lần gặp `"W"`, nó chạm `10` đúng ở sự kiện `"W"` thứ 10 và ta thoát ngay, không xử lý sự kiện nào sau đó.
3. **Không dừng sớm hơn.** Kiểm tra sau cập nhật nên sự kiện `"W"` thứ 10 vẫn được tính (counter = 10), đúng với ví dụ 3.
4. **Không dừng muộn hơn.** `break` đảm bảo các sự kiện sau `"W"` thứ 10 không ảnh hưởng `score`.
5. **Đủ sự kiện.** Nếu không đủ 10 sự kiện `"W"`, vòng lặp chạy hết và trả `[score, counter]` với `counter < 10`.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(events)`.

- Cách 1, 2, 4: `O(n)` thời gian (Cách 1, 2 có thể dừng sớm), `O(1)` bộ nhớ.
- Cách 3, 5: `O(n)` thời gian, `O(n)` bộ nhớ (lát cắt/`Counter`).

Best case: 10 sự kiện `"W"` ở đầu -> dừng sau 10 bước. Worst case: không bao giờ đủ 10 `"W"` -> duyệt hết `n` sự kiện.

## 12. Bộ test

Đối chiếu 5 cách + code nộp với ví dụ đề, biên (đúng 9, 10, 11 lần `"W"`; sự kiện sau điểm dừng) và 500 input ngẫu nhiên đúng constraints (độ dài tới 1000, các token hợp lệ, nhiều trường hợp đủ 10 `"W"`).

In [16]:
import random

solutions = [SolutionBrute(), SolutionDict(), SolutionCut(), SolutionReduce(), SolutionCounter(), Solution()]

fixed = [
    (["1", "4", "W", "6", "WD"], [12, 1]),
    (["WD", "NB", "0", "4", "4"], [10, 0]),
    (["W"] * 11, [0, 10]),
    (["W"] * 9, [0, 9]),
    (["W"] * 10 + ["6"], [0, 10]),                 # điểm 6 sau điểm dừng bị bỏ qua
    (["4"] * 3 + ["W"] * 10 + ["WD", "NB"], [12, 10]),
    (["0"], [0, 0]),
    (["6", "6", "6"], [18, 0]),
    (["WD"], [1, 0]),
    (["W", "WD", "W", "NB"], [2, 2]),
]
for ev, expected in fixed:
    for s in solutions:
        assert s.scoreValidator(list(ev)) == expected, (type(s).__name__, ev[:5], s.scoreValidator(list(ev)))

TOKENS = ["0", "1", "2", "3", "4", "6", "W", "WD", "NB"]
random.seed(39210)
for _ in range(500):
    n = random.randint(1, 1000)
    pw = random.choice([0.0, 0.02, 0.1, 0.3, 0.6])             # xác suất token là "W"
    ev = [("W" if random.random() < pw else random.choice([t for t in TOKENS if t != "W"])) for _ in range(n)]
    outs = [s.scoreValidator(list(ev)) for s in solutions]
    assert all(o == outs[0] for o in outs)
    score, counter = outs[0]
    assert 0 <= counter <= 10
    assert counter == min(10, ev.count("W"))
print("tất cả test OK")

tất cả test OK


### Test thuộc tính

(1) `counter == min(10, số "W")`. (2) Thêm sự kiện **vào cuối** mảng không đổi kết quả nếu đã dừng (`counter == 10`). (3) Tính cộng tính: nếu `counter < 10` ở nửa đầu thì kết quả của cả mảng bằng nửa đầu cộng với kết quả nửa sau (bắt đầu từ trạng thái trung gian), kiểm chứng bằng cách ghép các mảng không có `"W"`.

In [17]:
random.seed(3921)
for _ in range(300):
    n = random.randint(1, 200)
    ev = [random.choice(TOKENS) for _ in range(n)]
    base = Solution().scoreValidator(ev)
    assert base[1] == min(10, ev.count("W"))
    if base[1] == 10:
        extra = [random.choice(TOKENS) for _ in range(random.randint(1, 20))]
        assert Solution().scoreValidator(ev + extra) == base
    nw1 = [t for t in ev if t != "W"]
    nw2 = [t for t in ev[::-1] if t != "W"]
    s1, c1 = Solution().scoreValidator(nw1)
    s2, c2 = Solution().scoreValidator(nw2)
    assert c1 == 0 and c2 == 0 and s1 == s2 and Solution().scoreValidator(nw1 + nw2) == [s1 + s2, 0]
print("property test OK")

property test OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: không dừng khi `counter` chạm 10

Nếu bỏ `break`, các sự kiện sau điểm dừng vẫn được cộng vào `score` (ví dụ `["W"]*10 + ["6"]` cho `[6, 10]` thay vì `[0, 10]`).

In [18]:
def wrong_no_stop(events):
    score = counter = 0
    for t in events:
        if t == "W":
            counter += 1
        elif t in ("WD", "NB"):
            score += 1
        else:
            score += int(t)                                         # SAI: không bao giờ dừng
    return [score, counter]

ev = ["W"] * 10 + ["6"]
print(wrong_no_stop(ev), "!=", Solution().scoreValidator(ev))
assert wrong_no_stop(ev) == [6, 10] and Solution().scoreValidator(ev) == [0, 10]

[6, 10] != [0, 10]


### Lỗi 2: dừng khi `counter > 10` (quá muộn)

Điều kiện dừng là `counter == 10` **ngay khi chạm**. Nếu chỉ dừng khi `counter > 10`, ta xử lý thêm các sự kiện sau điểm dừng (và `counter` không bao giờ vượt 10 vì ta chỉ tăng 1 mỗi lần, nên vòng lặp chạy hết).

In [19]:
def wrong_gt(events):
    score = counter = 0
    for t in events:
        if t == "W":
            counter += 1
        elif t in ("WD", "NB"):
            score += 1
        else:
            score += int(t)
        if counter > 10:                                           # SAI: phải == 10
            break
    return [score, counter]

ev = ["W"] * 10 + ["6"]
print(wrong_gt(ev), "!=", Solution().scoreValidator(ev))
assert wrong_gt(ev) == [6, 10] and Solution().scoreValidator(ev) == [0, 10]

[6, 10] != [0, 10]


### Lỗi 3: coi `"WD"` như `"W"` (tăng counter)

`"WD"` chỉ là điểm thưởng `+1`; không làm tăng `counter`. Dùng `startswith("W")` nhầm lẫn hai loại.

In [20]:
def wrong_startswith(events):
    score = counter = 0
    for t in events:
        if t.startswith("W"):                                      # SAI: gồm cả "WD"
            counter += 1
        elif t == "NB":
            score += 1
        else:
            score += int(t)
        if counter == 10:
            break
    return [score, counter]

ev = ["1", "4", "W", "6", "WD"]
print(wrong_startswith(ev), "!=", Solution().scoreValidator(ev))
assert wrong_startswith(ev) == [11, 2] and Solution().scoreValidator(ev) == [12, 1]

[11, 2] != [12, 1]


### Lỗi 4: gọi `int()` cho mọi token

`int("WD")`, `int("W")`, `int("NB")` đều lỗi `ValueError`. Phải xử lý các token chữ cái trước.

In [21]:
def wrong_int_all(events):
    score = 0
    for t in events:
        score += int(t)                                            # SAI: token chữ cái
    return [score, 0]

try:
    wrong_int_all(["1", "W"])
except ValueError as e:
    print("ValueError:", e)
assert Solution().scoreValidator(["1", "W"]) == [1, 1]

ValueError: invalid literal for int() with base 10: 'W'


### Lỗi 5: quên `"NB"` (hoặc `"WD"`)

Chỉ xử lý một trong hai loại điểm thưởng làm mất điểm hoặc ném lỗi. Ví dụ 2 có cả `"WD"` lẫn `"NB"`.

In [22]:
def wrong_forget_nb(events):
    score = counter = 0
    for t in events:
        if t == "W":
            counter += 1
        elif t == "WD":                                            # SAI: thiếu "NB"
            score += 1
        elif t.isdigit():
            score += int(t)
        if counter == 10:
            break
    return [score, counter]

ev = ["WD", "NB", "0", "4", "4"]
print(wrong_forget_nb(ev), "!=", Solution().scoreValidator(ev))
assert wrong_forget_nb(ev) == [9, 0] and Solution().scoreValidator(ev) == [10, 0]

[9, 0] != [10, 0]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề dạng "xử lý lần lượt các sự kiện với luật" -> **simulation**: giữ vài biến trạng thái, một vòng `for`.
- Điều kiện dừng nằm ở **đề**: kiểm tra **ngay sau** cập nhật, rồi `break`.
- Token chữ/số lẫn lộn -> dùng `dict` bảng điểm hoặc kiểm tra `isdigit()` trước khi `int`.
- Tránh `startswith`/`in` trên chuỗi khi các token có thể là tiền tố của nhau (`"W"` và `"WD"`).
- Test biên: đúng `9`, `10`, `11` lần `"W"`; sự kiện điểm sau điểm dừng; mảng một phần tử.

## 15. Biến thể của bài

1. **Baseball Game** (682): mô phỏng điểm với các thao tác `"C"`, `"D"`, `"+"`.
2. **Calculate Money in Leetcode Bank** (1716), **Final Value of Variable After Performing Operations** (2011): mô phỏng cơ bản.
3. **Number of Students Unable to Eat Lunch**, **Time Needed to Buy Tickets** (2073): mô phỏng hàng đợi.
4. **Valid Parentheses** (20), **Evaluate Reverse Polish Notation** (150): xử lý token.
5. Biến thể: dừng khi `score` vượt một ngưỡng, hoặc dừng khi counter chạm `k` tùy ý; thêm loại sự kiện `"RO"` (run out) tăng counter.
6. Biến thể: trả về **chỉ số** sự kiện làm dừng trận đấu.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng `if` | `for` + `break` | `O(n)` | `O(1)` | Nộp bài |
| 2. `dict` điểm | bảng `SCORE`, `W` tăng counter | `O(n)` | `O(1)` | Nhiều loại token |
| 3. Tìm điểm cắt | tìm `cut`, rồi `sum` trên đoạn | `O(n)` | `O(n)` | Tách "dừng" khỏi "tính" |
| 4. `reduce` | trạng thái đóng băng khi dừng | `O(n)` | `O(1)` | Phong cách hàm |
| 5. `Counter` | đếm token trên đoạn đã cắt | `O(n)` | `O(n)` | Khi cần thống kê token |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().scoreValidator(["6", "W", "NB", "3"])` và `Solution().scoreValidator(["W"] * 12 + ["4"])` ra gì?

**Bài 2 (tính tay):** với `events = ["4","4","W","6","WD","W","1"]`, lập bảng `score` và `counter` qua từng bước.

**Bài 3 (điền TODO):** hoàn thành `stop_index(events)` trả về chỉ số (bắt đầu từ 1) của sự kiện làm `counter` chạm `10`, hoặc `-1` nếu không có.

**Bài 4 (tự viết lại):** viết `score_with_limit(events, limit)` dừng khi `counter` chạm `limit` (thay vì cố định `10`).

**Bài 5 (mini challenge):** viết `score_until_target(events, target)` dừng ngay khi `score >= target` (hoặc khi `counter` chạm 10); trả về `[score, counter, số sự kiện đã xử lý]`.

In [23]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().scoreValidator(["6", "W", "NB", "3"]), Solution().scoreValidator(["W"] * 12 + ["4"]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def stop_index_todo(events):
    counter = 0
    # TODO: duyệt events, tăng counter ở "W", trả về chỉ số (từ 1) khi counter == 10
    return -1

[10, 1] [0, 10]


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: dãy thứ hai có 12 sự kiện `"W"`, dừng ở sự kiện thứ 10. Bài 2: bảng như mục 4.

**Mức 2 (vừa):** Bài 3: `enumerate(events, 1)`. Bài 4: thay `10` bằng `limit` ở điều kiện dừng. Bài 5: kiểm tra `score >= target` sau mỗi cập nhật rồi `break`.

**Mức 3 (gần đáp án):** Bài 1: `[10, 1]` và `[0, 10]`. Bài 2: kết quả cuối `[16, 2]`: `4,4 -> 8; W -> counter 1; 6 -> 14; WD -> 15; W -> counter 2; 1 -> 16`. Bài 3: `stop_index(["W"]*12) = 10`.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, 2
assert Solution().scoreValidator(["6", "W", "NB", "3"]) == [10, 1]
assert Solution().scoreValidator(["W"] * 12 + ["4"]) == [0, 10]
assert Solution().scoreValidator(["4", "4", "W", "6", "WD", "W", "1"]) == [16, 2]

# Bài 3
def stop_index(events):
    counter = 0
    for i, t in enumerate(events, 1):
        if t == "W":
            counter += 1
            if counter == 10:
                return i
    return -1
assert stop_index(["W"] * 12) == 10 and stop_index(["W"] * 9) == -1 and stop_index(["1", "W"] * 10) == 20

# Bài 4
def score_with_limit(events, limit):
    score = counter = 0
    for t in events:
        if t == "W":
            counter += 1
        elif t in ("WD", "NB"):
            score += 1
        else:
            score += int(t)
        if counter == limit:
            break
    return [score, counter]
assert score_with_limit(["1", "W", "W", "4"], 2) == [1, 2]
assert score_with_limit(["W"] * 15, 10) == Solution().scoreValidator(["W"] * 15)

# Bài 5
def score_until_target(events, target):
    score = counter = used = 0
    for t in events:
        used += 1
        if t == "W":
            counter += 1
        elif t in ("WD", "NB"):
            score += 1
        else:
            score += int(t)
        if counter == 10 or score >= target:
            break
    return [score, counter, used]
assert score_until_target(["4", "4", "6", "6"], 8) == [8, 0, 2]
assert score_until_target(["W"] * 12, 100) == [0, 10, 10]
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "mô phỏng sự kiện có điều kiện dừng":**

```python
score = counter = 0
for t in events:
    if t == "W":
        counter += 1
    elif t in ("WD", "NB"):
        score += 1
    else:
        score += int(t)
    if counter == 10:      # kiểm tra NGAY SAU cập nhật
        break
return [score, counter]
```

**Checklist trước khi submit:**

- [ ] `"W"` chỉ tăng `counter`; `"WD"`/`"NB"` cộng `1` điểm; chữ số cộng đúng giá trị?
- [ ] Dừng **ngay** khi `counter == 10` (không xử lý sự kiện kế tiếp)?
- [ ] Không dùng `startswith("W")` (nhầm `"WD"`) và không gọi `int()` trên token chữ cái?
- [ ] Trả về `[score, counter]` (đúng thứ tự)?
- [ ] Đã thử đúng `9`, `10`, `11` lần `"W"` và các ví dụ đề?
- [ ] Chữ ký đúng `scoreValidator(self, events: List[str]) -> List[int]`?